To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.7.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the [Alpaca dataset](https://huggingface.co/datasets/unsloth/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read our docs [here](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#training-on-completions-only-masking-out-inputs)

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Conversational.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("json", data_files="alpaca_dataset.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.08)
train_dataset = dataset["train"].map(formatting_prompts_func, batched=True)
eval_dataset = dataset["test"].map(formatting_prompts_func, batched=True)
dataset = dataset.map(formatting_prompts_func, batched = True,)

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [9]:
MAX_TOKENS = 4096

def is_short_enough(example):
    return len(tokenizer(example["text"]).input_ids) <= MAX_TOKENS

train_dataset = train_dataset.filter(is_short_enough)
eval_dataset = eval_dataset.filter(is_short_enough)

print(f"Train examples after filtering: {len(train_dataset)}")
print(f"Eval examples after filtering: {len(eval_dataset)}")

Filter:   0%|          | 0/425 [00:00<?, ? examples/s]

Filter:   0%|          | 0/38 [00:00<?, ? examples/s]

Train examples after filtering: 399
Eval examples after filtering: 37


In [5]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    dataset_num_proc = 1,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        eval_strategy = "steps",
        eval_steps = 20,
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/424 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/37 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
8.953 GB of memory reserved.


In [6]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 424 | Num Epochs = 3 | Total steps = 159
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
20,0.850900,0.684408
40,0.286500,0.380794
60,0.244100,0.341503
80,0.191300,0.326477
100,0.413600,0.316602
120,0.245200,0.310330
140,0.375700,0.305708


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [9]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1901.9158 seconds used for training.
31.7 minutes used for training.
Peak reserved memory = 8.953 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 61.478 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [10]:
model.save_pretrained("lora_model-2")
tokenizer.save_pretrained("lora_model-2")


('lora_model-2/tokenizer_config.json',
 'lora_model-2/special_tokens_map.json',
 'lora_model-2/chat_template.jinja',
 'lora_model-2/vocab.json',
 'lora_model-2/merges.txt',
 'lora_model-2/added_tokens.json',
 'lora_model-2/tokenizer.json')

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonacci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nContinue the fibonacci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\n13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [11]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

test_instruction = """You are a digital forensics assistant. You will be given a raw JSON artifact
(extracted file metadata OR a browser artifact record: cookie, transport-security entry, or
NEL record) from a forensic evidence extraction tool.

Your job: analyze it and output ONLY a JSON object with exactly these top-level fields:

{
  "entities": {
    "ips": [],
    "domains": [],
    "emails": [],
    "hashes": [],
    "urls": [],
    "usernames": [],
    "person_names": [],
    "phone_numbers": [],
    "addresses": []
  },
  "classification": {
    "risk": "benign" | "suspicious" | "malicious",
    "reason": "short justification, 1-3 sentences, specific to this artifact"
  },
  "artifact_summary": "one or two sentence human-readable summary of this artifact for an investigator"
}

Rules:
- Output ONLY the JSON object above. No markdown fences, no preamble, no explanation outside the JSON.
- Leave entity arrays empty ([]) if nothing of that type is present. Do not invent values.
- "reason" MUST reference at least one specific field/value from the input (e.g. a filename, timestamp, tool name, size, or attribute actually present in the artifact). NEVER use generic filler phrases like "no suspicious indicators", "metadata consistent", or "no signs of tampering" on their own -- if the file is unremarkable, say what specifically makes it unremarkable (e.g. "modified and created timestamps are 13 seconds apart, consistent with normal editing/save behavior" rather than just "no signs of tampering").
- Use "suspicious" or "malicious" only when there is a concrete indicator (e.g. typosquatting,
  known-bad TLD pattern, timestamp anomalies, hidden+system attributes on unexpected files,
  abnormal expiry dates, mismatched tool provenance). Default to "benign" when nothing stands out.
- ctime (NTFS file creation record) commonly becomes LATER than the modified timestamp whenever a
  file is copied, downloaded, extracted, or moved to a new location/drive -- this is normal filesystem
  behavior, NOT inherently suspicious by itself. Do not flag "ctime after modified" alone as suspicious.
  Only flag timestamp patterns when combined with additional concrete indicators (e.g. hidden+system
  attributes on an unexpected file type, a typosquatted filename, an executable in a sensitive system
  directory, or a modified timestamp that is itself impossible/inconsistent, such as predating the
  software/tool that created it).`
- For hashed/opaque domain fields (e.g. Chrome HSTS SHA256-hashed domains), put the hash value in
  "hashes", not "domains".
- CRITICAL: only include values in "entities" that literally, verbatim appear in the input artifact. NEVER invent, guess, or fabricate hashes, IPs, domains, or any other identifier that is not explicitly present in the given input. If a hash/IP/etc field is not present in the input, leave the corresponding array empty -- do not fill it with a plausible-looking placeholder.
- Filename-based typosquatting (character substitution, added/removed letters, homoglyphs mimicking a legitimate system process or brand name) is a strong indicator on its own, independent of NTFS attributes. Flag it even when hidden/system/archive attributes look otherwise normal.
"""  # reuse the same instruction text used during dataset creation

test_cases = {
    "clearly_benign": '''{"path": "C:\\\\Users\\\\Asus\\\\Documents\\\\notes.txt", "name": "notes.txt", "extension": ".txt", "is_file": true, "is_directory": false, "is_symlink": false, "size_bytes": 512, "size_human": "512 B", "time_modified": "2026-06-15 10:00:00 UTC", "time_accessed": "2026-07-01 09:00:00 UTC", "time_ctime": "2026-06-15 10:00:05 UTC", "ctime_meaning": "file creation time (NTFS \\u2014 stored in MFT)", "platform": "Windows", "is_readable": true, "is_writable": true, "ntfs_attributes": {"readonly": false, "hidden": false, "system": false, "archive": true, "encrypted": false, "compressed": false, "sparse": false}, "ntfs_mft_info": {"mft_file_index": 1234, "hard_link_count": 1, "volume_serial": "0xf02e1c2f"}, "format": "TXT", "encoding": "utf-8", "line_count": 10, "content": "shopping list: milk, eggs, bread"}''',

    "browser_cookie": '''{"name": "session_token", "domain": ".github.com", "path": "/", "value": "abc123", "created": "2026-07-10 08:00:00 UTC", "expires": "2026-08-10 08:00:00 UTC", "last_accessed": "2026-07-10 09:00:00 UTC", "secure": true, "httponly": true, "samesite": 1, "persistent": true, "priority": 1, "source_scheme": 2}''',

    "normal_ctime_gap": '''{"path": "C:\\\\Users\\\\Asus\\\\Downloads\\\\report.pdf", "name": "report.pdf", "extension": ".pdf", "is_file": true, "is_directory": false, "is_symlink": false, "size_bytes": 204800, "size_human": "200 KB", "time_modified": "2025-01-10 12:00:00 UTC", "time_accessed": "2026-07-10 09:00:00 UTC", "time_ctime": "2026-07-05 14:30:00 UTC", "ctime_meaning": "file creation time (NTFS \\u2014 stored in MFT)", "platform": "Windows", "is_readable": true, "is_writable": true, "ntfs_attributes": {"readonly": false, "hidden": false, "system": false, "archive": true, "encrypted": false, "compressed": false, "sparse": false}, "ntfs_mft_info": {"mft_file_index": 5678, "hard_link_count": 1, "volume_serial": "0xf02e1c2f"}, "format": "PDF", "page_count": 3, "author": "Jane Smith", "producer": "Microsoft Word"}''',

    "ambiguous_borderline": '''{"path": "C:\\\\Users\\\\Asus\\\\AppData\\\\Local\\\\Temp\\\\update_helper.exe", "name": "update_helper.exe", "extension": ".exe", "is_file": true, "is_directory": false, "is_symlink": false, "size_bytes": 45000, "size_human": "43.9 KB", "time_modified": "2026-07-09 03:00:00 UTC", "time_accessed": "2026-07-09 03:05:00 UTC", "time_ctime": "2026-07-09 03:00:00 UTC", "ctime_meaning": "file creation time (NTFS \\u2014 stored in MFT)", "platform": "Windows", "is_readable": true, "is_writable": true, "ntfs_attributes": {"readonly": false, "hidden": false, "system": false, "archive": true, "encrypted": false, "compressed": false, "sparse": false}, "ntfs_mft_info": {"mft_file_index": 9012, "hard_link_count": 1, "volume_serial": "0xf02e1c2f"}, "format": "PE32", "author": "unavailable"}''',

    "clear_malicious_typosquat": '''{"path": "C:\\\\Windows\\\\System32\\\\explorer0.exe", "name": "explorer0.exe", "extension": ".exe", "is_file": true, "is_directory": false, "is_symlink": false, "size_bytes": 98304, "size_human": "96 KB", "time_modified": "2026-07-08 02:00:00 UTC", "time_accessed": "2026-07-09 22:00:00 UTC", "time_ctime": "2026-07-09 21:59:00 UTC", "ctime_meaning": "file creation time (NTFS \\u2014 stored in MFT)", "platform": "Windows", "is_readable": true, "is_writable": true, "ntfs_attributes": {"readonly": false, "hidden": true, "system": true, "archive": true, "encrypted": false, "compressed": false, "sparse": false}, "ntfs_mft_info": {"mft_file_index": 3456, "hard_link_count": 1, "volume_serial": "0xf02e1c2f"}, "format": "PE32", "author": "unavailable"}''',

    "transport_security_hashed_domain": '''{"domain": "9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08", "expiry": "2027-01-01 00:00:00 UTC", "include_subdomains": true, "mode": "force-https"}''',
}
for label, test_input in test_cases.items():
    print(f"\\n===== {label} =====")
    inputs = tokenizer(
        [alpaca_prompt.format(test_instruction, test_input, "")],
        return_tensors="pt"
    ).to("cuda")
    output = model.generate(**inputs, max_new_tokens=400, use_cache=True)
    decoded = tokenizer.batch_decode(output)[0]
    # print only the response part after "### Response:"
    print(decoded.split("### Response:")[-1].replace("<|im_end|>", "").strip())

\n===== clearly_benign =====
{"entities": {"ips": [], "domains": [], "emails": [], "hashes": [], "urls": [], "usernames": [], "person_names": [], "phone_numbers": [], "addresses": []}, "classification": {"risk": "benign", "reason": "Standard text file with no suspicious indicators, modified and accessed timestamps consistent with normal user activity, contains personal shopping list information."}, "artifact_summary": "Text file 'notes.txt' containing a personal shopping list, located in a standard user document folder, with typical file attributes and timestamps."}
\n===== browser_cookie =====
{"entities": {"ips": [], "domains": ["github.com"], "emails": [], "hashes": [], "urls": [], "usernames": [], "person_names": [], "phone_numbers": [], "addresses": []}, "classification": {"risk": "suspicious", "reason": "The session_token cookie has a long-lived expiration date of 1 month (from July to August), which is unusual for GitHub's authentication mechanism."}, "artifact_summary": "GitHub

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is a famous tall tower in Paris?

### Input:


### Response:
One of the most famous and iconic tall towers in Paris is the Eiffel Tower. Standing at 324 meters (1,063 feet) tall, this wrought iron tower is a symbol of the city and a must-see attraction for tourists from all over the world.<|end_of_text|>

You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("llama_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

In [12]:
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:43<00:00, 111.87s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:00<00:00, 60.21s/it]


Unsloth: Merge process complete. Saved to `/content/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9950-mix-53618c5 (app-b9950-mix-53618c5-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/Qwen2.5-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully

{'save_directory': 'model_gguf',
 'gguf_directory': 'model_gguf_gguf',
 'gguf_files': ['model_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'model_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [13]:
import os

os.listdir("model_gguf_gguf")

['Qwen2.5-3B-Instruct.Q4_K_M.gguf', 'Modelfile']

In [14]:
!zip -j qwen_model.zip model_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf

  adding: Qwen2.5-3B-Instruct.Q4_K_M.gguf (deflated 2%)


In [15]:
from google.colab import files
files.download("qwen_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
!zip -r model_gguf.zip model_gguf

  adding: model_gguf/ (stored 0%)
  adding: model_gguf/special_tokens_map.json (deflated 69%)
  adding: model_gguf/tokenizer_config.json (deflated 83%)
  adding: model_gguf/model-00001-of-00002.safetensors (deflated 20%)
  adding: model_gguf/generation_config.json (deflated 33%)
  adding: model_gguf/tokenizer.json (deflated 81%)
  adding: model_gguf/chat_template.jinja (deflated 71%)
  adding: model_gguf/model.safetensors.index.json (deflated 96%)
  adding: model_gguf/.cache/ (stored 0%)
  adding: model_gguf/.cache/huggingface/ (stored 0%)
  adding: model_gguf/.cache/huggingface/download/ (stored 0%)
  adding: model_gguf/.cache/huggingface/download/model-00001-of-00002.safetensors.metadata (deflated 29%)
  adding: model_gguf/.cache/huggingface/download/model-00002-of-00002.safetensors.metadata (deflated 31%)
  adding: model_gguf/.cache/huggingface/download/model.safetensors.index.json.metadata (deflated 25%)
  adding: model_gguf/.cache/huggingface/.gitignore (stored 0%)
  adding: model

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>